# Посимвольная токенизация

В этом ноутбуке каждая буква, знак препинания и пробел являются отдельными токенами.

Задача: обучить Simple RNN, однослойную LSTM, многослойную LSTM, BiLSTM и mini-GPT для генерации текста.

## 1. Подготовка окружения

Первая ячейка проверяет зависимости и при необходимости устанавливает их. Если окружение уже подготовлено, установка пропускается.

In [ ]:
import math
import random
import re
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from urllib.request import urlopen


def ensure_package(package_name, import_name=None):
    import importlib.util

    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "-q"])


ensure_package("torch")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 2. Загрузка корпуса

Используется текст *Alice's Adventures in Wonderland* из Project Gutenberg. При отсутствии сети используется встроенный резервный фрагмент.

In [ ]:
GUTENBERG_URL = "https://www.gutenberg.org/files/11/11-0.txt"
FALLBACK_TEXT = """
Alice was beginning to get very tired of sitting by her sister on the bank,
and of having nothing to do. Once or twice she had peeped into the book her
sister was reading, but it had no pictures or conversations in it.
"And what is the use of a book," thought Alice, "without pictures or
conversation?"
So she was considering in her own mind, as well as she could, for the hot day
made her feel very sleepy and stupid, whether the pleasure of making a
daisy-chain would be worth the trouble of getting up and picking the daisies.
"""


def load_text(max_chars=180_000):
    try:
        with urlopen(GUTENBERG_URL, timeout=20) as response:
            raw = response.read().decode("utf-8", errors="ignore")
        start = raw.find("CHAPTER I.")
        end = raw.find("*** END OF THE PROJECT GUTENBERG")
        if start != -1 and end != -1:
            raw = raw[start:end]
    except Exception:
        raw = FALLBACK_TEXT * 300

    text = re.sub(r"\s+", " ", raw).strip()
    return text[:max_chars]


text = load_text()
print(f"Corpus length: {len(text):,} characters")
print(text[:500])


## 3. Токенизация и параметры эксперимента

In [ ]:
class CharTokenizer:
    def __init__(self, text):
        self.itos = sorted(set(text))
        self.stoi = {ch: i for i, ch in enumerate(self.itos)}
        self.vocab_size = len(self.itos)

    def encode(self, value):
        return [self.stoi[ch] for ch in value if ch in self.stoi]

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)


tokenizer = CharTokenizer(text)
BLOCK_SIZE = 96
BATCH_SIZE = 64
EPOCHS = 3
LEARNING_RATE = 3e-3
MAX_BATCHES = 120
GENERATE_TOKENS = 300


## 4. Датасет, модели и функции обучения

In [ ]:
class TokenDataset(Dataset):
    def __init__(self, ids, block_size):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.block_size = block_size

    def __len__(self):
        return max(0, len(self.ids) - self.block_size)

    def __getitem__(self, idx):
        chunk = self.ids[idx : idx + self.block_size + 1]
        return chunk[:-1], chunk[1:]


class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=192, kind="rnn", num_layers=1, bidirectional=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        rnn_cls = nn.RNN if kind == "rnn" else nn.LSTM
        self.rnn = rnn_cls(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
        )
        directions = 2 if bidirectional else 1
        self.proj = nn.Linear(hidden_dim * directions, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.rnn(emb)
        return self.proj(out)


class CausalSelfAttention(nn.Module):
    def __init__(self, emb_dim, n_heads, block_size, dropout=0.1):
        super().__init__()
        assert emb_dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = emb_dim // n_heads
        self.qkv = nn.Linear(emb_dim, 3 * emb_dim)
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        b, t, c = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:, :, :t, :t] == 0, float("-inf"))
        weights = self.dropout(F.softmax(scores, dim=-1))
        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(b, t, c)
        return self.proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, n_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.attn = CausalSelfAttention(emb_dim, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ff = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.GELU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=128, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(emb_dim, n_heads, block_size, dropout) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(emb_dim)
        self.head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        b, t = x.shape
        positions = torch.arange(t, device=x.device)
        x = self.token_emb(x) + self.pos_emb(positions)[None, :, :]
        x = self.blocks(x)
        return self.head(self.ln(x))


def train_model(model, train_loader, val_loader, epochs=3, lr=3e-3, max_batches=120):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for batch_idx, (x, y) in enumerate(train_loader):
            if batch_idx >= max_batches:
                break
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        val_loss = evaluate(model, val_loader, max_batches=30)
        history.append({"epoch": epoch, "train_loss": sum(train_losses) / len(train_losses), "val_loss": val_loss})
        print(f"epoch={epoch} train_loss={history[-1]['train_loss']:.3f} val_loss={val_loss:.3f}")

    return history


@torch.no_grad()
def evaluate(model, loader, max_batches=30):
    model.eval()
    losses = []
    for batch_idx, (x, y) in enumerate(loader):
        if batch_idx >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        losses.append(loss.item())
    return sum(losses) / len(losses)


@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=120, temperature=0.8):
    model.eval()
    ids = tokenizer.encode(prompt)
    if len(ids) == 0:
        ids = [0]
    x = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        context = x[:, -model.block_size :] if hasattr(model, "block_size") else x[:, -BLOCK_SIZE:]
        logits = model(context)[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_id], dim=1)

    return tokenizer.decode(x[0].tolist())


## 5. Подготовка train/validation выборок

In [ ]:
ids = tokenizer.encode(text)
vocab_size = tokenizer.vocab_size
split = int(len(ids) * 0.9)
train_ids, val_ids = ids[:split], ids[split:]

train_dataset = TokenDataset(train_ids, BLOCK_SIZE)
val_dataset = TokenDataset(val_ids, BLOCK_SIZE)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f"Vocabulary size: {vocab_size}")
print(f"Train sequences: {len(train_dataset):,}; validation sequences: {len(val_dataset):,}")


## 6. Обучение и сравнение моделей

Для ускорения лабораторного запуска число эпох и батчей ограничено. При наличии GPU можно увеличить `EPOCHS` и `MAX_BATCHES`.

In [ ]:
def build_models():
    return {
        "Simple RNN": RNNLanguageModel(vocab_size, kind="rnn", num_layers=1),
        "LSTM 1 layer": RNNLanguageModel(vocab_size, kind="lstm", num_layers=1),
        "LSTM 2 layers": RNNLanguageModel(vocab_size, kind="lstm", num_layers=2),
        "BiLSTM": RNNLanguageModel(vocab_size, kind="lstm", num_layers=1, bidirectional=True),
        "Mini GPT": MiniGPT(vocab_size, block_size=BLOCK_SIZE),
    }


results = []
trained_models = {}
for name, model in build_models().items():
    print(f"\nTraining {name}")
    history = train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE, max_batches=MAX_BATCHES)
    val_loss = history[-1]["val_loss"]
    results.append({"model": name, "val_loss": val_loss, "perplexity": math.exp(min(val_loss, 20))})
    trained_models[name] = model

results


## 7. Генерация текста

Для каждой обученной модели генерируется продолжение одного и того же начального промпта.

In [ ]:
prompt = "Alice was"
for name, model in trained_models.items():
    print("=" * 80)
    print(name)
    print(generate(model, tokenizer, prompt, max_new_tokens=GENERATE_TOKENS, temperature=0.8))
